In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Check GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


### Function to load sample questions from dataset

In [3]:
from datasets import load_dataset
def RAG_Bench_ds_loader(dataset_name, defined_set,testing):
  ds = load_dataset("galileo-ai/ragbench", dataset_name, split=defined_set)
  if testing == True:
    ds = ds[:30]
  df = pd.DataFrame(ds)
  all_docs = []
  for _, row in df.iterrows():
      for doc_pos, doc_text in enumerate(row["documents"]):
          all_docs.append({
              "doc_id":   f"{row['id']}_d{doc_pos}",
              "row_id":   str(row["id"]),
              "doc_pos":  doc_pos,          # 0 to len(documents)-1
              "text":     doc_text.strip(),
              "question": row["question"],
              "response": row["response"]
          })

  docs_df = pd.DataFrame(all_docs)
  print(f"Total documents: {len(docs_df)}")
  return docs_df



### Load Huggingface token

In [4]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
groq_token = user_secrets.get_secret("GROQ_TOKEN")
openrouter_token = user_secrets.get_secret("OPENROUTER_TOKEN")

# Set it as an environment variable (what HuggingFace libraries expect)
import os
os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN set ✓")

HF_TOKEN set ✓


In [5]:
!pip install requests

### call the function to load sample

In [6]:
import pandas as pd

docs_df = RAG_Bench_ds_loader("delucionqa","test",True)

README.md: 0.00B [00:00, ?B/s]

delucionqa/train-00000-of-00001.parquet:   0%|          | 0.00/4.23M [00:00<?, ?B/s]

delucionqa/validation-00000-of-00001.par(…):   0%|          | 0.00/528k [00:00<?, ?B/s]

delucionqa/test-00000-of-00001.parquet:   0%|          | 0.00/562k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/182 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/184 [00:00<?, ? examples/s]

Total documents: 90


### installing required libraries

In [7]:
!pip install -qU \
    "opentelemetry-api>=1.36.0,<1.39.0" \
    "opentelemetry-sdk>=1.36.0,<1.39.0" \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-groq \
    langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.3 MB/s eta 0:00:00


### Phase 1 : Splitting the documents into Chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,           # characters (not tokens)
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],  # tries each in order
)


chunks = []
for _, doc in docs_df.iterrows():
    splits = splitter.split_text(doc["text"])
    for chunk_idx, chunk_text in enumerate(splits):
        chunks.append({
            "chunk_id":     f"{doc['doc_id']}_c{chunk_idx}",
            "doc_id":       doc["doc_id"],
            "row_id":       doc["row_id"],
            "doc_pos":      doc["doc_pos"],
            "chunk_idx":    chunk_idx,
            "total_chunks": len(splits),
            "text":         chunk_text,
        })


### Converting them to Langchain Documents

In [9]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Convert dicts → Document objects
# 'text' becomes page_content, everything else goes into metadata
documents = [
    Document(
        page_content=chunk["text"],
        metadata={
            "chunk_id"    : chunk["chunk_id"],
            "doc_id"      : chunk["doc_id"],
            "row_id"      : chunk["row_id"],
            "doc_pos"     : chunk["doc_pos"],
        }
    )
    for chunk in chunks
]

print(f"Converted {len(documents)} dicts → Document objects")
print(f"Sample page_content : {documents[0].page_content[:100]}")
print(f"Sample metadata     : {documents[0].metadata}")

Converted 640 dicts → Document objects
Sample page_content : Closing To close the tailgate, lift upward until both sides latch into place.  CAUTION: After closin
Sample metadata     : {'chunk_id': '114_d0_c0', 'doc_id': '114_d0', 'row_id': '114', 'doc_pos': 0}


### Loading the and configuring the embedding model

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

# DO NOT specify device—let it auto-detect GPU
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Embedding model loaded (auto-detected device)")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded (auto-detected device)


### Creating Vector DB with index and store the embeddings

In [11]:
CHROMA_PATH = "/kaggle/working/chroma_db"

# Build index here, then download it from the output panel
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="ragbench_delucion",
    persist_directory=CHROMA_PATH,
)

### RAG Pipeline

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import sys, os


def _add_ragbench_lib_to_path():
    for candidate in (os.getcwd(), os.path.join(os.getcwd(), "delucion_dataset")):
        if os.path.isdir(os.path.join(candidate, "ragbench_lib")) and candidate not in sys.path:
            sys.path.insert(0, candidate)
            return


_add_ragbench_lib_to_path()

from ragbench_lib.models import get_generation_llm
from ragbench_lib.generation_prompt import RAG_GENERATION_PROMPT

llm = get_generation_llm(openrouter_token)

# ── Prompt ───────────────────────────────────────────────────────────────
prompt = RAG_GENERATION_PROMPT

# ── Retriever (replaces your retrieve() function) ───────────────────────────
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7},
)

# ── format_docs (replaces your build_context()) ─────────────────────────────
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# ── LCEL chain (replaces your generate() function) ──────────────────────────
rag_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
question = docs_df["question"].iloc[19]

answer = rag_chain.invoke(question)

print(f"Question     : {question}")
print(f"\nGenerated    :\n{answer}")
print(f"\nGround Truth :\n{docs_df['response'].iloc[19]}")

Question     : how to calculate the gross trailer weight?

Generated    :
According to the context, the recommended way to measure Gross Trailer Weight (GTW) is to put your fully loaded trailer on a vehicle scale, where the entire weight of the trailer must be supported by the scale.

Ground Truth :
To calculate the gross trailer weight (GTW), the recommended way is to put your fully loaded trailer on a vehicle scale where the entire weight of the trailer must be supported by the scale. The GTW is the weight of the trailer plus the weight of all cargo, consumables, and equipment (permanent or temporary) loaded in or on the trailer in its "loaded and ready for operation" condition. This measurement should be taken when the trailer is fully loaded and ready to be towed.


### Evaluators

### Evaluation: Llama 70B Annotation vs RAGBench Ground Truth

This section implements the RAGBench annotation prompt structure for Llama 70B and compares outputs against pre-computed RAGBench scores.

#### Step 1: Llama 70B Setup & Annotation Utilities

In [ ]:
from ragbench_lib.models import get_judge_llm
from ragbench_lib.chunking import get_sentences
from ragbench_lib.trace_eval import format_documents_with_keys
import json, re

# Initialize Llama 70B for annotation
llama_judge = get_judge_llm(openrouter_token)

print("✓ Initialized Llama 70B annotation utilities")


#### Step 2: RAGBench Annotation Prompt (Section 7.4)

In [ ]:
# The annotation prompt itself now lives in ragbench_lib.trace_eval.ANNOTATION_PROMPT_TEMPLATE
# (used internally by annotate_response_for_metrics below).
from ragbench_lib.trace_eval import ANNOTATION_PROMPT_TEMPLATE

print("✓ Loaded RAGBench annotation prompt (ragbench_lib.trace_eval)")


#### Step 3: Annotation Function with JSON Parsing

In [ ]:
from ragbench_lib.trace_eval import annotate_response_for_metrics as _annotate_response_for_metrics


def annotate_response_for_metrics(documents: list[str], question: str, response: str) -> dict:
    """Annotate YOUR RAG response to extract relevant and utilized sentence keys.
    Returns structured data for TRACe metric calculation.
    """
    return _annotate_response_for_metrics(llama_judge, documents, question, response)


print("✓ Annotation function ready")


In [ ]:
# ── TRACe Metric Calculation Functions (Paper Formulas) ────────────────────
from ragbench_lib.trace_eval import (
    compute_context_relevance as compute_context_relevance_from_annotation,
    compute_utilization as compute_utilization_from_annotation,
    compute_completeness as compute_completeness_from_annotation,
    compute_adherence as compute_adherence_from_annotation,
)

print("✓ TRACe metric functions defined")


In [18]:
# Ground truth extraction using VERIFIED field names from RAGBench
def extract_ragbench_ground_truth(num_samples=10):
    """
    Extract ground truth metrics from RAGBench DelucionQA.
    
    Field mapping (verified):
    - adherence_score: Boolean
    - relevance_score: Float (fraction of relevant sentences)
    - utilization_score: Float (fraction of utilized sentences)
    - completeness_score: Float (intersection of relevant & utilized)
    """
    ds = load_dataset("galileo-ai/ragbench", "delucionqa", split="test")
    
    ground_truth = []
    for i, sample in enumerate(ds):
        if i >= num_samples:
            break
        
        gt_sample = {
            # Identifiers
            'row_id': sample['id'],
            'question': sample['question'],
            'ground_truth_response': sample['response'],
            'documents': sample['documents'],
            
            # ✅ GROUND TRUTH METRICS (verified field names)
            'gt_adherence': 1.0 if sample['adherence_score'] else 0.0,  # Convert bool to float
            'gt_context_relevance': sample.get('relevance_score', None),
            'gt_utilization': sample.get('utilization_score', None),
            'gt_completeness': sample.get('completeness_score', None),
            
            # ✅ GROUND TRUTH ANNOTATIONS (for understanding)
            'gt_relevant_keys': sample.get('all_relevant_sentence_keys', []),
            'gt_utilized_keys': sample.get('all_utilized_sentence_keys', []),
        }
        
        ground_truth.append(gt_sample)
    
    return ground_truth

print("Extracting ground truth from RAGBench (DelucionQA)...")
gold_data = extract_ragbench_ground_truth(num_samples=10)
print(f"✓ Extracted {len(gold_data)} samples")

# Verify we have the data
if gold_data:
    print(f"\nSample 1 Ground Truth:")
    sample = gold_data[0]
    print(f"  Question: {sample['question'][:60]}...")
    print(f"  GT Adherence: {sample['gt_adherence']}")
    print(f"  GT Relevance: {sample['gt_context_relevance']}")
    print(f"  GT Utilization: {sample['gt_utilization']}")
    print(f"  GT Completeness: {sample['gt_completeness']}")

Extracting ground truth from RAGBench (DelucionQA)...
✓ Extracted 10 samples

Sample 1 Ground Truth:
  Question: What if I fail to latch the tailgate properly?...
  GT Adherence: 1.0
  GT Relevance: 0.1111111111111111
  GT Utilization: 0.1111111111111111
  GT Completeness: 1.0


In [19]:
print("\n" + "="*80)
print("STEP 1: RUN RAG & COMPUTE METRICS")
print("="*80 + "\n")

for i, gold_sample in enumerate(gold_data):
    question = gold_sample['question']
    documents = gold_sample['documents']
    
    print(f"[{i+1}/{len(gold_data)}] Q: {question[:70]}...")
    
    # ✅ STEP 1: RUN YOUR RAG PIPELINE
    try:
        my_response = rag_chain.invoke(question)
        gold_sample['my_response'] = my_response
        print(f"  My response: {my_response[:70]}...")
    except Exception as e:
        print(f"  ✗ RAG failed: {str(e)[:60]}")
        continue
    
    # ✅ STEP 2: ANNOTATE YOUR RESPONSE TO GET RELEVANT & UTILIZED KEYS
    print(f"  Annotating response...", end=" ")
    annotation = annotate_response_for_metrics(documents, question, my_response)
    
    if not annotation['success']:
        print(f"✗ Failed: {annotation.get('error', 'unknown')[:50]}")
        gold_sample['annotation'] = annotation
        continue
    
    print("✓")
    gold_sample['annotation'] = annotation
    
    # Show what the judge extracted
    print(f"    Relevant keys: {annotation['relevant_keys']}")
    print(f"    Utilized keys: {annotation['utilized_keys']}")
    
    # ✅ STEP 3: COMPUTE TRACE METRICS FROM ANNOTATION
    gold_sample['computed_context_relevance'] = compute_context_relevance_from_annotation(
        documents, annotation
    )
    gold_sample['computed_utilization'] = compute_utilization_from_annotation(
        documents, annotation
    )
    gold_sample['computed_completeness'] = compute_completeness_from_annotation(annotation)
    gold_sample['computed_adherence'] = compute_adherence_from_annotation(annotation)
    
    # Display computed metrics
    print(f"    Computed Metrics:")
    print(f"      Context Relevance: {gold_sample['computed_context_relevance']:.4f}")
    print(f"      Utilization:       {gold_sample['computed_utilization']:.4f}")
    print(f"      Completeness:      {gold_sample['computed_completeness']:.4f}")
    print(f"      Adherence:         {gold_sample['computed_adherence']:.4f}")
    print()

print("✓ All responses processed\n")


STEP 1: RUN RAG & COMPUTE METRICS

[1/10] Q: What if I fail to latch the tailgate properly?...
  My response: If you fail to latch the tailgate properly, it could result in damage ...
  Annotating response... ✓
    Relevant keys: ['0c', '2d']
    Utilized keys: ['0c']
    Computed Metrics:
      Context Relevance: 0.1111
      Utilization:       0.0556
      Completeness:      0.5000
      Adherence:         1.0000

[2/10] Q: What kind of safety features are implemented in this car?...
  My response: Based on the context, the safety features mentioned are:

1. Restraint...
  Annotating response... ✓
    Relevant keys: ['0a', '1a', '2a', '2f']
    Utilized keys: ['0a', '2a', '2f']
    Computed Metrics:
      Context Relevance: 0.2500
      Utilization:       0.1875
      Completeness:      0.7500
      Adherence:         0.0000

[3/10] Q: When will the Automatic SOS be triggered?...
  My response: The Automatic SOS will be triggered if the vehicle's airbags deploy....
  Annotating resp

#### Step 6: Summary & Next Steps

In [20]:
print("\n" + "="*100)
print("COMPUTED METRICS vs GROUND TRUTH DEVIATION ANALYSIS")
print("="*100 + "\n")

# Build comparison table
comparison_rows = []

for i, gold in enumerate(gold_data):
    
    # Skip if annotation failed
    if not gold.get('annotation', {}).get('success', False):
        print(f"[Sample {i+1}] Skipped (annotation failed)")
        continue
    
    # ✅ Extract computed metrics
    comp_rel = gold.get('computed_context_relevance', 0)
    comp_util = gold.get('computed_utilization', 0)
    comp_comp = gold.get('computed_completeness', 0)
    comp_adh = gold.get('computed_adherence', 0)
    
    # ✅ Extract ground truth metrics (using VERIFIED field names)
    gt_rel = gold.get('gt_context_relevance', None)
    gt_util = gold.get('gt_utilization', None)
    gt_comp = gold.get('gt_completeness', None)
    gt_adh = gold.get('gt_adherence', None)
    
    # ✅ Calculate deviations (absolute difference)
    dev_rel = abs(comp_rel - gt_rel) if gt_rel is not None else None
    dev_util = abs(comp_util - gt_util) if gt_util is not None else None
    dev_comp = abs(comp_comp - gt_comp) if gt_comp is not None else None
    dev_adh = abs(comp_adh - gt_adh) if gt_adh is not None else None
    
    comparison_rows.append({
        'Sample': i + 1,
        'Question': gold['question'][:35] + '...',
        
        # Context Relevance
        'Comp_Rel': f"{comp_rel:.4f}",
        'GT_Rel': f"{gt_rel:.4f}" if gt_rel is not None else "N/A",
        'Δ_Rel': f"{dev_rel:.4f}" if dev_rel is not None else "N/A",
        
        # Utilization
        'Comp_Util': f"{comp_util:.4f}",
        'GT_Util': f"{gt_util:.4f}" if gt_util is not None else "N/A",
        'Δ_Util': f"{dev_util:.4f}" if dev_util is not None else "N/A",
        
        # Completeness
        'Comp_Comp': f"{comp_comp:.4f}",
        'GT_Comp': f"{gt_comp:.4f}" if gt_comp is not None else "N/A",
        'Δ_Comp': f"{dev_comp:.4f}" if dev_comp is not None else "N/A",
        
        # Adherence
        'Comp_Adh': f"{comp_adh:.4f}",
        'GT_Adh': f"{gt_adh:.4f}" if gt_adh is not None else "N/A",
        'Δ_Adh': f"{dev_adh:.4f}" if dev_adh is not None else "N/A",
    })

# Display table
df_comparison = pd.DataFrame(comparison_rows)
display(df_comparison)

# Aggregate statistics
print("\n" + "="*100)
print("AGGREGATE DEVIATION STATISTICS")
print("="*100 + "\n")

# Extract numeric deviations (filter out "N/A")
rel_devs = [float(r['Δ_Rel']) for r in comparison_rows if r['Δ_Rel'] != 'N/A']
util_devs = [float(r['Δ_Util']) for r in comparison_rows if r['Δ_Util'] != 'N/A']
comp_devs = [float(r['Δ_Comp']) for r in comparison_rows if r['Δ_Comp'] != 'N/A']
adh_devs = [float(r['Δ_Adh']) for r in comparison_rows if r['Δ_Adh'] != 'N/A']

print("Mean Deviation (↓ lower = better agreement):")
print(f"  Context Relevance:  {np.mean(rel_devs) if rel_devs else 0:.4f}  ± {np.std(rel_devs) if rel_devs else 0:.4f}")
print(f"  Utilization:        {np.mean(util_devs) if util_devs else 0:.4f}  ± {np.std(util_devs) if util_devs else 0:.4f}")
print(f"  Completeness:       {np.mean(comp_devs) if comp_devs else 0:.4f}  ± {np.std(comp_devs) if comp_devs else 0:.4f}")
print(f"  Adherence:          {np.mean(adh_devs) if adh_devs else 0:.4f}  ± {np.std(adh_devs) if adh_devs else 0:.4f}")

print("\n" + "-"*100)
print("Interpretation Guide:")
print("  Δ < 0.05  → Excellent match (your computation ≈ RAGBench ground truth)")
print("  Δ 0.05-0.15  → Good match")
print("  Δ > 0.15  → Mismatch (check: LLM annotation quality, metric formula, or field names)")
print("-"*100)


COMPUTED METRICS vs GROUND TRUTH DEVIATION ANALYSIS



,Sample,Question,Comp_Rel,GT_Rel,Δ_Rel,Comp_Util,GT_Util,Δ_Util,Comp_Comp,GT_Comp,Δ_Comp,Comp_Adh,GT_Adh,Δ_Adh
0,1,What if I fail to latch the tailgat...,0.1111,0.1111,0.0000,0.0556,0.1111,0.0555,0.5000,1.0000,0.5000,1.0000,1.0000,0.0000
1,2,What kind of safety features are im...,0.2500,0.2500,0.0000,0.1875,0.2500,0.0625,0.7500,1.0000,0.2500,0.0000,1.0000,1.0000
2,3,When will the Automatic SOS be trig...,0.0244,0.0244,0.0000,0.0244,0.0244,0.0000,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000
3,4,What happens if I accidentally push...,0.0820,0.0820,0.0000,0.0328,0.0656,0.0328,0.4000,0.8000,0.4000,1.0000,1.0000,0.0000
4,5,What is the DEF?...,0.1875,0.0625,0.1250,0.0000,0.0625,0.0625,0.0000,1.0000,1.0000,0.0000,1.0000,1.0000
5,6,What may cause erratic or noisy per...,0.0857,0.1143,0.0286,0.0857,0.1143,0.0286,1.0000,1.0000,0.0000,1.0000,1.0000,0.0000
6,7,how to calculate the gross trailer ...,0.2727,0.2727,0.0000,0.0909,0.2727,0.1818,0.3333,1.0000,0.6667,1.0000,1.0000,0.0000
7,8,What can the ASIST button do?...,0.2174,0.4348,0.2174,0.0000,0.4348,0.4348,0.0000,1.0000,1.0000,0.0000,1.0000,1.0000
8,9,What does the Door Off Mirror Kit d...,0.1739,0.1304,0.0435,0.0435,0.1304,0.0869,0.2500,1.0000,0.7500,1.0000,1.0000,0.0000
9,10,Can I manually activate or deactiva...,0.0811,0.0811,0.0000,0.0000,0.1351,0.1351,0.0000,0.6667,0.6667,0.0000,1.0000,1.0000



AGGREGATE DEVIATION STATISTICS

Mean Deviation (↓ lower = better agreement):
  Context Relevance:  0.0415  ± 0.0696
  Utilization:        0.1081  ± 0.1200
  Completeness:       0.5233  ± 0.3443
  Adherence:          0.4000  ± 0.4899

----------------------------------------------------------------------------------------------------
Interpretation Guide:
  Δ < 0.05  → Excellent match (your computation ≈ RAGBench ground truth)
  Δ 0.05-0.15  → Good match
  Δ > 0.15  → Mismatch (check: LLM annotation quality, metric formula, or field names)
----------------------------------------------------------------------------------------------------
